In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vit_timesformer_probe import TimeSformer, TimeSformerIBCALinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:3' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 01:32:34,485]::456791383::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:3


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)

In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# TimeSformer IBCA Block-wise Linear Probe -- UDIVA / MSE
# ============================================================
#
# Backbone is completely frozen.
# Only the independent probe heads are optimized.
#
# IMPORTANT:
#   Evaluation metric = MSE (lower is better).
#   Do NOT calculate/report 1-MSE.
#   O/C/E/A/N are also reported directly as MSE.
# ============================================================

PROBE_SEED = 42
PROBE_LR = 1e-3
WEIGHT_DECAY = 1e-4
PROBE_EPOCHS = 50
PATIENCE = 10

BACKBONE_CHECKPOINT = "./save_timesformer_folder/model_best_multiseed.pth"


def set_probe_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def probe_seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_probe_seed(PROBE_SEED)

generator = torch.Generator()
generator.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

backbone = TimeSformer().to(device)
checkpoint = torch.load(BACKBONE_CHECKPOINT, map_location=device)
state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
backbone.load_state_dict(state_dict, strict=True)

model = TimeSformerIBCALinearProbe(
    backbone=backbone,
    dim=768,
    num_outputs=5
).to(device)

backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
probe_trainable = sum(p.numel() for p in model.probe_heads.parameters() if p.requires_grad)

print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")
print(f"Metric: MSE | Patience: {PATIENCE}")
assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.MSELoss().to(device)
optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=PROBE_LR,
    weight_decay=WEIGHT_DECAY
)

probe_names = list(model.probe_heads.keys())
best_mse = {name: float("inf") for name in probe_names}
best_epoch = {name: -1 for name in probe_names}
best_mean_val_mse = float("inf")
no_improve_count = 0

for epoch in range(PROBE_EPOCHS):
    # -------------------- Train probe heads only --------------------
    model.train()
    train_sq_sum = {name: 0.0 for name in probe_names}
    train_count = 0

    for audio, fullshot, big_five_data in tqdm(
        train_dataloader,
        desc=f"Probe Train {epoch + 1}/{PROBE_EPOCHS}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
        if big_five_data.ndim == 1:
            big_five_data = big_five_data.unsqueeze(0)
        audio = audio.to(device)

        optimizer.zero_grad()
        outputs = model(fullshot, audio)

        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }

        total_loss = sum(losses.values())
        total_loss.backward()
        optimizer.step()

        train_count += big_five_data.numel()
        for name in probe_names:
            train_sq_sum[name] += torch.square(
                outputs[name].detach() - big_five_data
            ).sum().item()

    train_mse = {
        name: train_sq_sum[name] / train_count
        for name in probe_names
    }

    # -------------------- Validation --------------------
    model.eval()
    val_sq_sum = {name: 0.0 for name in probe_names}
    val_trait_sq_sum = {
        name: np.zeros(5, dtype=np.float64)
        for name in probe_names
    }
    val_count = 0
    val_sample_count = 0

    with torch.no_grad():
        for audio, fullshot, big_five_data in tqdm(
            val_dataloader,
            desc=f"Probe Valid {epoch + 1}/{PROBE_EPOCHS}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
            if big_five_data.ndim == 1:
                big_five_data = big_five_data.unsqueeze(0)
            audio = audio.to(device)

            outputs = model(fullshot, audio)
            val_count += big_five_data.numel()
            val_sample_count += big_five_data.shape[0]

            for name in probe_names:
                sq_err = torch.square(outputs[name] - big_five_data)
                val_sq_sum[name] += sq_err.sum().item()
                val_trait_sq_sum[name] += (
                    sq_err.sum(dim=0).detach().cpu().numpy()
                )

    val_mse = {
        name: val_sq_sum[name] / val_count
        for name in probe_names
    }
    val_trait_mse = {
        name: val_trait_sq_sum[name] / val_sample_count
        for name in probe_names
    }

    # Each probe independently tracks its own lowest validation MSE.
    for name in probe_names:
        if val_mse[name] < best_mse[name]:
            best_mse[name] = val_mse[name]
            best_epoch[name] = epoch + 1

    # Global early stopping criterion: mean validation MSE across probes.
    mean_val_mse = float(np.mean([val_mse[name] for name in probe_names]))

    if mean_val_mse < best_mean_val_mse:
        best_mean_val_mse = mean_val_mse
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 132)
    print(
        f"Epoch {epoch + 1:03d} | "
        f"Mean Val MSE={mean_val_mse:.6f} | "
        f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
        f"Patience={no_improve_count}/{PATIENCE}"
    )

    for name in probe_names:
        o_mse, c_mse, e_mse, a_mse, n_mse = val_trait_mse[name]
        print(
            f"{name:<12} | "
            f"Train MSE {train_mse[name]:.6f} | "
            f"Val MSE {val_mse[name]:.6f} | "
            f"Best MSE {best_mse[name]:.6f} | "
            f"O MSE {o_mse:.6f} | C MSE {c_mse:.6f} | "
            f"E MSE {e_mse:.6f} | A MSE {a_mse:.6f} | "
            f"N MSE {n_mse:.6f}"
        )

    if no_improve_count >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch + 1} | "
            f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
            f"Patience={no_improve_count}/{PATIENCE}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Final Probe Summary -- MSE only
# ============================================================
print("\n" + "=" * 72)
print("FINAL TIMESFORMER IBCA LINEAR PROBE RESULTS -- UDIVA / MSE")
print("=" * 72)
print(
    f"{'Representation':<18}"
    f"{'Best Epoch':>12}"
    f"{'Best MSE':>16}"
)
print("-" * 72)

for name in probe_names:
    print(
        f"{name:<18}"
        f"{best_epoch[name]:>12d}"
        f"{best_mse[name]:>16.6f}"
    )

print("=" * 72)

del model
del backbone
del optimizer
del train_dataloader
del val_dataloader
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299
Backbone trainable parameters: 0
Probe trainable parameters:    69953
Metric: MSE | Patience: 10


Probe Valid 1/50: 100%|██████████| 191/191 [00:20<00:00,  9.21it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 001 | Mean Val MSE=1.129787 | Best Mean Val MSE=1.129787 | Patience=0/10
block1       | Train MSE 1.132278 | Val MSE 1.122466 | Best MSE 1.122466 | O MSE 0.908710 | C MSE 1.239987 | E MSE 1.030969 | A MSE 1.383723 | N MSE 1.048943
block2       | Train MSE 1.132808 | Val MSE 1.122076 | Best MSE 1.122076 | O MSE 0.903511 | C MSE 1.242467 | E MSE 1.032526 | A MSE 1.382063 | N MSE 1.049815
block3       | Train MSE 1.134984 | Val MSE 1.124034 | Best MSE 1.124034 | O MSE 0.907417 | C MSE 1.245807 | E MSE 1.031209 | A MSE 1.384976 | N MSE 1.050761
block4       | Train MSE 1.131989 | Val MSE 1.120886 | Best MSE 1.120886 | O MSE 0.899587 | C MSE 1.244206 | E MSE 1.026520 | A MSE 1.383589 | N MSE 1.050527
block5       | Train MSE 1.134913 | Val MSE 1.121416 | Best MSE 1.121416 | O MSE 0.906306 | C MSE 1.249808 | E MSE 1.021288 | A MSE 1.380183 | N MSE 1.0494

Probe Valid 2/50: 100%|██████████| 191/191 [00:20<00:00,  9.28it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 002 | Mean Val MSE=1.130631 | Best Mean Val MSE=1.129787 | Patience=1/10
block1       | Train MSE 1.110432 | Val MSE 1.129563 | Best MSE 1.122466 | O MSE 0.982555 | C MSE 1.222236 | E MSE 1.027009 | A MSE 1.346430 | N MSE 1.069586
block2       | Train MSE 1.113374 | Val MSE 1.131317 | Best MSE 1.122076 | O MSE 0.985765 | C MSE 1.223966 | E MSE 1.035016 | A MSE 1.349009 | N MSE 1.062829
block3       | Train MSE 1.114746 | Val MSE 1.134214 | Best MSE 1.124034 | O MSE 0.993418 | C MSE 1.230352 | E MSE 1.033635 | A MSE 1.348972 | N MSE 1.064693
block4       | Train MSE 1.116280 | Val MSE 1.129926 | Best MSE 1.120886 | O MSE 0.989825 | C MSE 1.227114 | E MSE 1.026824 | A MSE 1.340285 | N MSE 1.065583
block5       | Train MSE 1.118532 | Val MSE 1.125296 | Best MSE 1.121416 | O MSE 0.993162 | C MSE 1.216687 | E MSE 1.017267 | A MSE 1.336836 | N MSE 1.0625

...


------------------------------------------------------------------------------------------------------------------------------------
Epoch 009 | Mean Val MSE=1.161792 | Best Mean Val MSE=1.129787 | Patience=8/10
block1       | Train MSE 1.018629 | Val MSE 1.161566 | Best MSE 1.122466 | O MSE 0.865664 | C MSE 1.351175 | E MSE 1.078926 | A MSE 1.385730 | N MSE 1.126335
block2       | Train MSE 1.036983 | Val MSE 1.167480 | Best MSE 1.122076 | O MSE 0.867535 | C MSE 1.372565 | E MSE 1.102455 | A MSE 1.360967 | N MSE 1.133877
block3       | Train MSE 1.040354 | Val MSE 1.173577 | Best MSE 1.124034 | O MSE 0.881444 | C MSE 1.390144 | E MSE 1.106273 | A MSE 1.361897 | N MSE 1.128127
block4       | Train MSE 1.042005 | Val MSE 1.173916 | Best MSE 1.120886 | O MSE 0.883606 | C MSE 1.391163 | E MSE 1.110451 | A MSE 1.358019 | N MSE 1.126340
block5       | Train MSE 1.043717 | Val MSE 1.165256 | Best MSE 1.121416 | O MSE 0.881546 | C MSE 1.376792 | E MSE 1.094876 | A MSE 1.355783 | N MSE 1.1172

Probe Valid 10/50: 100%|██████████| 191/191 [00:21<00:00,  8.69it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 010 | Mean Val MSE=1.209451 | Best Mean Val MSE=1.129787 | Patience=9/10
block1       | Train MSE 1.013249 | Val MSE 1.195978 | Best MSE 1.122466 | O MSE 0.888550 | C MSE 1.287544 | E MSE 1.110696 | A MSE 1.449091 | N MSE 1.244007
block2       | Train MSE 1.033918 | Val MSE 1.185130 | Best MSE 1.122076 | O MSE 0.887896 | C MSE 1.287698 | E MSE 1.113906 | A MSE 1.412638 | N MSE 1.223511
block3       | Train MSE 1.037807 | Val MSE 1.186956 | Best MSE 1.124034 | O MSE 0.903968 | C MSE 1.299860 | E MSE 1.110957 | A MSE 1.409778 | N MSE 1.210219
block4       | Train MSE 1.039427 | Val MSE 1.179968 | Best MSE 1.120886 | O MSE 0.903943 | C MSE 1.307079 | E MSE 1.094698 | A MSE 1.401594 | N MSE 1.192525
block5       | Train MSE 1.041204 | Val MSE 1.182115 | Best MSE 1.121416 | O MSE 0.907586 | C MSE 1.307474 | E MSE 1.103447 | A MSE 1.399527 | N MSE 1.1925

Probe Valid 11/50: 100%|██████████| 191/191 [00:34<00:00,  5.56it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 011 | Mean Val MSE=1.130495 | Best Mean Val MSE=1.129787 | Patience=10/10
block1       | Train MSE 1.007101 | Val MSE 1.141142 | Best MSE 1.122466 | O MSE 0.947075 | C MSE 1.189704 | E MSE 1.069182 | A MSE 1.396714 | N MSE 1.103035
block2       | Train MSE 1.029719 | Val MSE 1.133414 | Best MSE 1.122076 | O MSE 0.912391 | C MSE 1.202599 | E MSE 1.085463 | A MSE 1.367535 | N MSE 1.099084
block3       | Train MSE 1.033794 | Val MSE 1.135332 | Best MSE 1.124034 | O MSE 0.912895 | C MSE 1.218313 | E MSE 1.091699 | A MSE 1.361295 | N MSE 1.092457
block4       | Train MSE 1.035315 | Val MSE 1.129118 | Best MSE 1.120886 | O MSE 0.907019 | C MSE 1.212417 | E MSE 1.090918 | A MSE 1.353704 | N MSE 1.081529
block5       | Train MSE 1.037251 | Val MSE 1.124348 | Best MSE 1.121416 | O MSE 0.913564 | C MSE 1.194559 | E MSE 1.082092 | A MSE 1.350541 | N MSE 1.080